# **Схемы: activation patching задачи IOI**

Задача IOI (indirect object identification): в предложении «When John and Mary went to the shops, Mary gave the bag to» модель должна продолжить « John» — имя, которое не повторялось. [Wang et al., 2023](https://arxiv.org/abs/2211.00593) нашли в GPT-2 small схему из 26 голов внимания, которая решает эту задачу. В этой практике вы повторите главный прием их работы — activation patching — и увидите, на каких слоях и в каких головах лежит информация, нужная для ответа.

Задания пронумерованы, ответы на них вводятся на сайте курса в уроке «Схемы: Практика».

## **Подготовка окружения**
Хватит CPU; на GPU прогон быстрее.

In [ ]:
!pip install transformer-lens==3.9.0 -q

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)  # здесь только прямые проходы
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = HookedTransformer.from_pretrained("gpt2-small", device=device)

## **Данные: чистые и испорченные промпты**

В каждом чистом промпте два имени: IO (indirect object, правильный ответ) и S (subject, упомянут дважды). В испорченном промпте второе упоминание S заменено на IO, поэтому правильного ответа у модели больше нет. Пары отличаются ровно одним токеном — так patching покажет, где модель хранит именно эту разницу. У Wang et al. (2023) для path patching испорченный промпт устроен иначе: в том же шаблоне все три упоминания имен заменены другими случайными именами. От способа порчи входа результат patching зависит (Zhang и Nanda, 2024), поэтому числа этой тетради относятся к порче «второе упоминание S заменено на IO».

In [ ]:
names = [(" John", " Mary"), (" Tom", " James"), (" Dan", " Sid"), (" Martin", " Amy")]
places = [("shops", "bag"), ("park", "ball")]

clean, corrupted, io_names, s_names = [], [], [], []
for a, b in names:
    for place, obj in places:
        for first, second in [(a, b), (b, a)]:
            # чистый промпт: второе упоминание (S2) — второй из названных, ответ (IO) — первый
            clean.append(f"When{first} and{second} went to the {place},{second} gave the {obj} to")
            # испорченный: S2 заменено на другое имя, остальное то же
            corrupted.append(f"When{first} and{second} went to the {place},{first} gave the {obj} to")
            io_names.append(first)
            s_names.append(second)

clean_tokens = model.to_tokens(clean)
corrupted_tokens = model.to_tokens(corrupted)
io_ids = torch.tensor([model.to_single_token(n) for n in io_names], device=device)
s_ids = torch.tensor([model.to_single_token(n) for n in s_names], device=device)

print(len(clean), "пар промптов, длина", clean_tokens.shape[1], "токенов")
print(clean[0])
print(corrupted[0])
print(model.to_str_tokens(clean_tokens[0]))

## **Метрика: логит-разность**

Меряем $\mathrm{logit}(IO) - \mathrm{logit}(S)$ на последней позиции, средней по 16 промптам. Положительная — модель предпочитает правильное имя.

In [ ]:
def logit_diff(logits):
    """Средняя по промптам разность логитов IO и S на последней позиции."""
    last = logits[:, -1]
    return ...  # Ваш код здесь


clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits = model(corrupted_tokens)
clean_ld, corrupted_ld = logit_diff(clean_logits).item(), logit_diff(corrupted_logits).item()
print(f"чистые промпты: {clean_ld:.2f}, испорченные: {corrupted_ld:.2f}")

**Задание 1. Чему равна средняя логит-разность на чистых промптах? Ответ округлите до сотых.**

**Задание 2. Чему она равна на испорченных промптах? Ответ округлите до сотых.**

## **Patching остаточного потока**

Прогоним испорченные промпты и на одном слое и одной позиции подставим остаточный поток из чистого прогона. Если логит-разность вернулась, нужная информация в этом месте лежала. Результат нормируем: 0 — как у испорченного промпта, 1 — как у чистого.

In [ ]:
def recovered(logits):
    """Доля логит-разности, которую вернул патч: 0 — как у испорченного промпта, 1 — как у чистого."""
    return (logit_diff(logits).item() - corrupted_ld) / (clean_ld - corrupted_ld)


n_layers, n_pos = model.cfg.n_layers, clean_tokens.shape[1]
resid = torch.zeros(n_layers, n_pos)
for layer in range(n_layers):
    for pos in range(n_pos):
        def patch(act, hook, pos=pos):
            act[:, pos] = clean_cache[hook.name][:, pos]  # чистая активация на одной позиции
            return act
        out = model.run_with_hooks(corrupted_tokens, fwd_hooks=[(utils.get_act_name("resid_pre", layer), patch)])
        resid[layer, pos] = recovered(out)

labels = [f"{t}_{i}" for i, t in enumerate(model.to_str_tokens(clean_tokens[0]))]
plt.figure(figsize=(10, 4))
plt.imshow(resid, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
plt.xticks(range(n_pos), labels, rotation=90)
plt.ylabel("layer (resid_pre)")
plt.colorbar(label="recovered logit difference")
plt.show()

s2_pos, end_pos = 10, n_pos - 1
print("позиция S2:", [round(v, 2) for v in resid[:, s2_pos].tolist()])
print("последняя позиция:", [round(v, 2) for v in resid[:, end_pos].tolist()])

**Задание 3. Начиная с какого слоя patching остаточного потока на последней позиции возвращает больше половины логит-разности?**

**Задание 4. Где лежит нужная информация на слоях до этого?**

## **Patching голов внимания**

Теперь подставим выход одной головы внимания (`hook_z`) из чистого прогона на всех позициях — 12 слоев по 12 голов. Голова с большим положительным эффектом переносит нужную информацию, с отрицательным — работает против ответа.

In [ ]:
n_heads = model.cfg.n_heads
heads = torch.zeros(n_layers, n_heads)
for layer in range(n_layers):
    for head in range(n_heads):
        def patch(z, hook, head=head):
            z[:, :, head] = clean_cache[hook.name][:, :, head]  # чистый выход одной головы на всех позициях
            return z
        out = model.run_with_hooks(corrupted_tokens, fwd_hooks=[(utils.get_act_name("z", layer), patch)])
        heads[layer, head] = recovered(out)

plt.figure(figsize=(6, 5))
plt.imshow(heads, cmap="RdBu", vmin=-0.6, vmax=0.6)
plt.xlabel("head")
plt.ylabel("layer")
plt.colorbar(label="recovered logit difference")
plt.show()

# Ваш код здесь: выведите пять голов с наибольшим эффектом и голову с наименьшим (самым отрицательным)

**Задание 5. Чему равен самый отрицательный эффект головы — доля логит-разности, которую возвращает подмена одной головы? Ответ округлите до сотых.**

**Задание 6. Какая это голова?** В работе Wang et al. такие головы названы negative name movers: они пишут в остаточный поток против имени, на которое смотрят.

## **Заключение**

- Информация, которая отличает чистый промпт от испорченного, сначала лежит на позиции второго упоминания S, а на средних слоях перемещается на последнюю позицию: patching остаточного потока показывает это перемещение по слоям.
- Patching голов выделяет немногие головы из 144; их роли совпадают с найденными Wang et al.: головы, которые находят повтор имени, головы S-inhibition на слоях 7–8, name movers на слоях 9–10 и negative name movers, работающие против ответа.
- Patching показывает, где лежит информация, нужная для этого различия на этих промптах. Схема из этого — гипотеза, которую проверяют на других шаблонах промптов, другой метрике и другом способе порчи входа.